# Qwen3-ASR 0.6B LoRA Levantine Custom Run: Palestine + Jordan + North Levantine
This notebook is a near plug-and-play copy of the Whisper Medium Levantine custom run, with only the Qwen3-ASR 0.6B changes needed for processor/model loading, instruction-style labels, and safer generation decoding.


## 1. Config

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path("/home/MohammadNabulsi/whisper")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import (
    NOTEBOOK_PATH,
    RUN_DIR,
    SMOKE_NOTEBOOK_PATH,
    config_snapshot,
    ensure_run_layout,
    make_config,
    setup_logging,
)

RUN_SMOKE_TEST = True
RUN_BASELINE_BEFORE_TRAIN = True
RUN_POST_TRAIN_EVAL = True

config = make_config(
    smoke_mode=RUN_SMOKE_TEST,
    num_train_epochs=50,
    run_baseline_before_train=RUN_BASELINE_BEFORE_TRAIN,
    run_post_train_eval=RUN_POST_TRAIN_EVAL,
)
ensure_run_layout()
log_path = setup_logging()
print(json.dumps(config_snapshot(config), ensure_ascii=False, indent=2))
print("Notebook path:", SMOKE_NOTEBOOK_PATH if RUN_SMOKE_TEST else NOTEBOOK_PATH)
print("Log path:", log_path)


## 2. Build Filtered Mini Manifests

In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import prepare_manifests

manifest_state = prepare_manifests(config)
selection_summary = manifest_state["selection_summary"]
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))


## 3. Baseline Qwen3-ASR 0.6B Predictions


In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import load_qwen_bundle, run_predictions

baseline_bundle = load_qwen_bundle(config)
baseline_test_metrics = None
if config.run_baseline_before_train:
    baseline_test_metrics = run_predictions(
        manifest_state["test_rows"],
        config,
        baseline_bundle,
        name="baseline_test_predictions",
    )
print(json.dumps(baseline_test_metrics, ensure_ascii=False, indent=2))


## 4. Train LoRA Adapters


In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import train_model

training_summary = train_model(config, manifest_state)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


## 5. Tuned Validation and Test Predictions


In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import load_qwen_bundle, run_predictions

tuned_bundle = load_qwen_bundle(config, adapter_path=training_summary["best_checkpoint"])
val_prediction_metrics = run_predictions(
    manifest_state["val_rows"],
    config,
    tuned_bundle,
    name="tuned_val_predictions",
) if config.run_post_train_eval else None
test_prediction_metrics = run_predictions(
    manifest_state["test_rows"],
    config,
    tuned_bundle,
    name="tuned_test_predictions",
) if config.run_post_train_eval else None
print("Validation metrics:")
print(json.dumps(val_prediction_metrics, ensure_ascii=False, indent=2))
print("Test metrics:")
print(json.dumps(test_prediction_metrics, ensure_ascii=False, indent=2))


## 6. Summary Report

In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import write_summary_report

summary_report = write_summary_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
print(json.dumps(summary_report, ensure_ascii=False, indent=2))


## 7. Integrity Report

In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import write_integrity_report

integrity_report = write_integrity_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
print(json.dumps(integrity_report, ensure_ascii=False, indent=2))
if integrity_report.get("prediction_health", {}).get("test_predictions", {}).get("object_dump_predictions", 0):
    raise RuntimeError("Decode guard failed: found object-dump predictions in test output.")
if integrity_report.get("prediction_health", {}).get("test_predictions", {}).get("empty_predictions", 0):
    raise RuntimeError("Integrity check failed: found empty predictions in test output.")
print("QWEN3-ASR 0.6B LEVANTINE RUN INTEGRITY CHECK PASSED")
